# Nemotron Local CV (Kaggle / RTX PRO 6000)

Runs `nemotron_local_cv.host_metric` against the 600-row stratified val split. Mirrors the host scorer's prompt + extraction + verification byte-for-byte.

Attach: this code dataset, the base model, and the trained adapter (either as another Dataset or as a kernel-source pointing at the training kernel's output).


In [ ]:
# Set memory env BEFORE torch import (cannot be changed after).
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Glob-discover instead of hardcoding — Kaggle's path conventions vary
# (`/kaggle/input/datasets/<owner>/<slug>/` for our newer dataset,
#  `/kaggle/input/<slug>/` for older public ones).
import glob, sys
matches = glob.glob("/kaggle/input/**/blackwell_setup.py", recursive=True)
assert matches, "code dataset not attached (no blackwell_setup.py found under /kaggle/input)"
CODE_ROOT = os.path.dirname(matches[0])
print(f"CODE_ROOT = {CODE_ROOT}")
sys.path.insert(0, CODE_ROOT)

import nemotron_local_cv  # noqa: F401
import nemotron_train  # noqa: F401
print("nemotron_local_cv + nemotron_train imported.")


In [ ]:
# Pre-model-load Blackwell setup. The mamba_ssm stub MUST be installed before
# AutoModelForCausalLM.from_pretrained loads modeling_nemotron_h.py.
from blackwell_setup import (  # noqa: E402
    set_cuda_alloc_env,
    set_blackwell_ptxas_env,
    install_mamba_ssm_stub,
)
set_cuda_alloc_env()
set_blackwell_ptxas_env()
install_mamba_ssm_stub()
print("CUDA alloc env + Blackwell ptxas env + mamba_ssm stub ready.")


In [ ]:
# peft + vllm for adapter loading/inference. Some Kaggle images include vllm,
# others do not; install from offline packages to keep this kernel deterministic.
# Internet must be OFF for competition compliance, so `--no-index --find-links`
# is mandatory.
import glob, subprocess, sys
matches = glob.glob("/kaggle/input/**/offline_packages", recursive=True)
assert matches, "offline_packages dir not found under /kaggle/input"
OFFLINE_DIR = matches[0]
print(f"OFFLINE_DIR = {OFFLINE_DIR}")
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "--no-cache-dir",
    "--no-index", "--find-links", OFFLINE_DIR,
    "peft", "vllm",
])
print("peft + vllm installed from offline packages.")


In [ ]:
from pathlib import Path

# Use the kernel-metadata.json-attached mount directly (faster than kagglehub.download).
BASE_MODEL_PATH = Path("/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1")
assert BASE_MODEL_PATH.is_dir(), f"base model not mounted at {BASE_MODEL_PATH}"
print(f"BASE_MODEL_PATH: {BASE_MODEL_PATH}")
print("  files:", sorted(p.name for p in BASE_MODEL_PATH.iterdir())[:6], "...")

# val_split_600.csv ships inside the code dataset (see scripts/sync_code_to_kaggle.py).
VAL_SPLIT = Path(CODE_ROOT) / "data" / "val_split_600.csv"
assert VAL_SPLIT.exists(), VAL_SPLIT


In [ ]:
# Locate the trained adapter. Two ways to attach it to this kernel:
#   (a) Add the training kernel as a kernel-source: adapter lives at
#       /kaggle/input/<train-kernel-slug>/adapter/
#   (b) Upload it as a private Dataset and attach: /kaggle/input/<dataset>/...
# Both code paths are covered by the glob below.
import glob

cands = glob.glob("/kaggle/input/**/adapter_config.json", recursive=True)
assert cands, "No adapter_config.json found in /kaggle/input/**. Attach the trained-adapter source."
ADAPTER_DIR = str(Path(cands[0]).parent)
print(f"ADAPTER_DIR: {ADAPTER_DIR}")


In [ ]:
# Run local CV via the host-metric mirror.
import pandas as pd
import tempfile
from pathlib import Path
from nemotron_local_cv import host_metric

val_df = pd.read_csv(VAL_SPLIT)
print(f"val rows: {len(val_df)}")
print(val_df["label"].value_counts().to_dict())

with tempfile.TemporaryDirectory() as tmp:
    tmp_path = Path(tmp)
    val_df.to_csv(tmp_path / "test.csv", index=False)
    host_metric.configure(model_path=str(BASE_MODEL_PATH), data_path=tmp_path)

    solution = val_df[["id", "answer"]]
    submission_stub = pd.DataFrame([{"prediction": ADAPTER_DIR}])

    accuracy = host_metric.score(
        solution=solution,
        submission=submission_stub,
        row_id_column_name="id",
        debug=True,
    )

print(f"\n=== overall accuracy: {accuracy:.4f} ===")


In [ ]:
# Per-label breakdown joined from the debug CSV produced above.
from pathlib import Path

debug = Path("/kaggle/working/debug_predictions.csv")
if debug.exists():
    pred = pd.read_csv(debug, dtype="str").fillna({"extracted_prediction": "NOT_FOUND"})
    merge = val_df.astype({"id": "str"}).merge(pred.astype({"id": "str"}), on="id", how="left")
    merge["is_correct"] = merge.apply(
        lambda r: host_metric.verify(str(r.answer), str(r.extracted_prediction)),
        axis=1,
    )
    print(merge.groupby("label").is_correct.agg(["mean", "count"]).to_string())
    merge.to_csv("/kaggle/working/debug_predictions_with_labels.csv", index=False)
else:
    print("debug_predictions.csv not found — re-run with debug=True")
